# AirShift — Final Test Evaluation

This notebook evaluates the final XGBoost model on the completely unseen test period.

The final model was trained using the combined training and validation periods, while the test period was kept separate throughout model development and hyperparameter tuning.

The evaluation focuses on the model's ability to detect future air quality deterioration events on unseen data.


## 1. Load the Dataset

In [ ]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


In [10]:
DATA_PATH = Path("../data/processed/airshift_labeled.csv")
MODEL_PATH = Path("../models/xgboost_final.joblib")

df = pd.read_csv(DATA_PATH)
df["datetime"] = pd.to_datetime(df["datetime"])

final_xgb = joblib.load(MODEL_PATH)

print("Dataset shape:", df.shape)
print("Final model loaded successfully.")

Dataset shape: (418381, 100)
Final model loaded successfully.


In [11]:
TEST_START = "2016-01-01 00:00:00"

test_mask = df["datetime"] >= TEST_START

test_df = df.loc[test_mask].copy()

print("Test dataset shape:", test_df.shape)
print("Test date range:", test_df["datetime"].min(), "to", test_df["datetime"].max())

Test dataset shape: (121900, 100)
Test date range: 2016-01-01 00:00:00 to 2017-02-28 17:00:00


## 2. Define Test Features and Target

The test features and target are separated for the final evaluation.

The identifier and datetime columns are excluded from the model features, while the `Deterioration` column is used as the target variable.


In [12]:
TARGET = "Deterioration"
EXCLUDED_COLUMNS = ["Deterioration", "No", "datetime"]

X_test = test_df.drop(columns=EXCLUDED_COLUMNS)
y_test = test_df[TARGET]

print("Test features shape:", X_test.shape)
print("Test target shape:", y_test.shape)
print("Number of features:", X_test.shape[1])

Test features shape: (121900, 97)
Test target shape: (121900,)
Number of features: 97


### Findings

The final XGBoost model was evaluated on the completely unseen 2016–2017 test period.

The model achieved performance comparable to the validation results, with no major degradation on the unseen test data. This indicates that the selected model configuration maintained consistent performance when evaluated on a later time period.

The final test results provide the main performance reference for the AirShift model before moving to model interpretation and deployment.


## 3. Final Test Predictions

The final XGBoost model is used to generate predictions on the completely unseen test period.

No model fitting, hyperparameter tuning, or threshold optimization is performed on the test data.


In [13]:
# Generate test predictions
y_test_pred = final_xgb.predict(X_test)
y_test_prob = final_xgb.predict_proba(X_test)[:, 1]

print("Test predictions generated successfully.")

Test predictions generated successfully.


## 4. Test Set Evaluation

The final XGBoost model is evaluated on the held-out test set covering observations from January 2016 to February 2017.

The test set was not used during model development or hyperparameter tuning, providing an unbiased evaluation of the final model's performance on unseen data.

The following metrics are calculated:

- **Accuracy:** Overall proportion of correct predictions.
- **Precision:** Proportion of predicted deterioration events that were actual deterioration events.
- **Recall:** Proportion of actual deterioration events correctly detected by the model.
- **F1 Score:** Harmonic mean of precision and recall.
- **ROC-AUC:** Measures the model's ability to distinguish between deterioration and non-deterioration cases across classification thresholds.
- **PR-AUC:** Measures performance across precision and recall and is particularly useful for evaluating the model's ability to identify deterioration events.

In [14]:
test_metrics = {
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred),
    "Recall": recall_score(y_test, y_test_pred),
    "F1": f1_score(y_test, y_test_pred),
    "ROC-AUC": roc_auc_score(y_test, y_test_prob),
    "PR-AUC": average_precision_score(y_test, y_test_prob)
}

test_results = pd.DataFrame(
    [test_metrics],
    index=["Final XGBoost"]
)

test_results.round(4)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
Final XGBoost,0.7368,0.7244,0.7315,0.7279,0.8215,0.8186
